# Tasks 1–3: FullSampleGloria (first 5,000 rows)

Reproduces Tasks 1, 2, and 3 on the new **FullSampleGloria** datasets.  
Only the **first 5,000 rows** of each file are used for testing/debugging.

**Input datasets**
- `FullSampleGloria_Pat_GlinerLabels_16042026.parquet`
- `FullSampleGloria_Link_PmidOa_16042026.parquet`
- `FullSampleGloria_Pmed_GlinerLabels_16042026.parquet`

**Outputs** saved to `output/fullsample_outputs/` and `visualizations/fullsample_visualizations/`.

## Imports and Paths

In [1]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

ROOT = Path("/Users/gloria/Desktop/Hiwi")

PAT_PATH  = ROOT / "data/raw/FullSampleGloria_Pat_GlinerLabels_16042026.parquet"
LINK_PATH = ROOT / "data/raw/FullSampleGloria_Link_PmidOa_16042026.parquet"
PMED_PATH = ROOT / "data/raw/FullSampleGloria_Pmed_GlinerLabels_16042026.parquet"

OUT_DIR = ROOT / "output/fullsample_outputs"
VIZ_DIR = ROOT / "visualizations/fullsample_visualizations"

NROWS = 5_000

## Data Loading

In [2]:
# Create output directories if they do not exist
for d in [OUT_DIR, VIZ_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"Ready: {d}")

Ready: /Users/gloria/Desktop/Hiwi/output/fullsample_outputs
Ready: /Users/gloria/Desktop/Hiwi/visualizations/fullsample_visualizations


## Task 1: Identify Focal Terms

Find terms that appear both in a patent and in its cited scientific papers.

In [3]:
# Step 1: Load patent GLiNER labels (first 5,000 rows)
pat_raw = pl.read_parquet(PAT_PATH, n_rows=NROWS)
print(f"Patent label rows loaded : {len(pat_raw):,}")
print(f"Columns                  : {pat_raw.columns}")
pat_raw.head(3)

Patent label rows loaded : 5,000
Columns                  : ['patent_id', 'term', 'label', 'GrantedDate', 'patent_type', 'patent_date', 'num_claims', 'withdrawn']


patent_id,term,label,GrantedDate,patent_type,patent_date,num_claims,withdrawn
str,str,str,date,str,date,f64,f64
"""10000000""","""detection""","""Machine Activity""",2018-06-19,"""utility""",2018-06-19,20.0,0.0
"""10000000""","""ladar""","""Medical Device""",2018-06-19,"""utility""",2018-06-19,20.0,0.0
"""10000000""","""system""","""Medical Device""",2018-06-19,"""utility""",2018-06-19,20.0,0.0


In [4]:
# Steps 2-3: Keep patent_id + term, group by (patent_id, term), count freq_in_patent
pat_terms = (
    pat_raw.select(["patent_id", "term"])
    .group_by(["patent_id", "term"])
    .agg(pl.len().alias("freq_in_patent"))
)
print(f"Unique patents in patent terms : {pat_terms['patent_id'].n_unique():,}")
print(f"Patent (id, term) rows         : {len(pat_terms):,}")
pat_terms.head(5)

Unique patents in patent terms : 53
Patent (id, term) rows         : 4,158


patent_id,term,freq_in_patent
str,str,u32
"""10000454""","""effective""",1
"""10000454""","""pparδ""",1
"""10000455""","""diastereomer""",2
"""10000466""","""r c""",1
"""10000472""","""coox""",1


In [5]:
# Steps 4-5: Load link table (first 5,000 rows), clean PMIDs
link_raw = pl.read_parquet(LINK_PATH, n_rows=NROWS)
print(f"Link rows loaded : {len(link_raw):,}")
print(f"Columns          : {link_raw.columns}")

# Drop rows without PMID; extract numeric PMID from URL with regex (\d+)$
link_clean = (
    link_raw.filter(pl.col("pmid").is_not_null())
    .with_columns(
        pl.col("pmid").str.extract(r"(\d+)$", 1).cast(pl.Int64).alias("pmid_num")
    )
    .filter(pl.col("pmid_num").is_not_null())
    .select([pl.col("patent_id"), pl.col("pmid_num").alias("pmid")])
    .unique()
)
print(f"\nCleaned patent–PMID links : {len(link_clean):,}")
print(f"Unique PMIDs in link table : {link_clean['pmid'].n_unique():,}")
link_clean.head(5)

Link rows loaded : 5,000
Columns          : ['patent_id', 'openalex', 'oaid', 'scipat', 'confscore', 'ppp_score', 'pmid', 'patent_date']

Cleaned patent–PMID links : 3,773
Unique PMIDs in link table : 3,667


patent_id,pmid
str,i64
"""10000532""",15569622
"""10000537""",2041760
"""10000539""",26121864
"""10000483""",833720
"""10000539""",23065523


In [6]:
# Steps 6-7: Load PubMed GLiNER labels (first 5,000 rows), keep pmid + term
pmed_raw = pl.read_parquet(PMED_PATH, n_rows=NROWS)
print(f"PubMed label rows loaded : {len(pmed_raw):,}")
print(f"Columns                  : {pmed_raw.columns}")

# Cast pmid (i32 in source) to Int64 to match link_clean join key
pmed_terms = pmed_raw.select([pl.col("pmid").cast(pl.Int64), "term"])
print(f"\nUnique PMIDs in paper terms : {pmed_terms['pmid'].n_unique():,}")
pmed_terms.head(5)

PubMed label rows loaded : 5,000
Columns                  : ['pmid', 'term', 'year', 'term_firstYr']

Unique PMIDs in paper terms : 34


pmid,term
i64,str
24670,"""11"""
24670,"""11"""
24670,"""11"""
24670,"""11"""
24670,"""11"""


In [7]:
# Step 8: Join paper terms with link table to attach patent_id
cited_joined = (
    pmed_terms
    .join(link_clean, on="pmid", how="inner")
)
print(f"Cited term rows after join : {len(cited_joined):,}")

# Step 9: Group by (patent_id, term) -> freq_in_cited_papers
cited_term_counts = (
    cited_joined
    .group_by(["patent_id", "term"])
    .agg(pl.len().alias("freq_in_cited_papers"))
)
print(f"Cited term count rows      : {len(cited_term_counts):,}")
cited_term_counts.head(5)

Cited term rows after join : 0
Cited term count rows      : 0


patent_id,term,freq_in_cited_papers
str,str,u32


In [8]:
# Step 10: Identify focal terms — inner join on (patent_id, term)
focal_terms = (
    pat_terms
    .join(cited_term_counts, on=["patent_id", "term"], how="inner")
    .rename({"term": "focal_term"})
)

if len(focal_terms) == 0:
    print("WARNING: No focal terms found in the first 5,000 rows.")
    print("This is likely due to row slicing — the sampled rows from patents, links,")
    print("and papers may not overlap. This does NOT indicate a pipeline failure.")
else:
    print(f"Focal term rows : {len(focal_terms):,}")
    print(focal_terms.head(10))

This is likely due to row slicing — the sampled rows from patents, links,
and papers may not overlap. This does NOT indicate a pipeline failure.


In [9]:
# Step 11: Save focal terms
focal_terms.write_parquet(OUT_DIR / "focal_terms_fullsample_5000.parquet")
focal_terms.write_csv(OUT_DIR / "focal_terms_fullsample_5000.csv")
print("Saved: focal_terms_fullsample_5000.parquet and .csv")

# Step 12: Sanity checks
print("\n=== Sanity Checks ===")
print(f"Patent rows loaded               : {len(pat_raw):,}")
print(f"Link rows loaded                 : {len(link_raw):,}")
print(f"PubMed rows loaded               : {len(pmed_raw):,}")
print(f"Unique patents in patent terms   : {pat_terms['patent_id'].n_unique():,}")
print(f"Unique PMIDs in link table       : {link_clean['pmid'].n_unique():,}")
print(f"Unique PMIDs in paper terms      : {pmed_terms['pmid'].n_unique():,}")
print(f"Cleaned patent–PMID links        : {len(link_clean):,}")
print(f"Focal term rows                  : {len(focal_terms):,}")
print(f"Unique patents with focal terms  : {focal_terms['patent_id'].n_unique():,}")
print(f"Unique focal terms               : {focal_terms['focal_term'].n_unique():,}")

Saved: focal_terms_fullsample_5000.parquet and .csv

=== Sanity Checks ===
Patent rows loaded               : 5,000
Link rows loaded                 : 5,000
PubMed rows loaded               : 5,000
Unique patents in patent terms   : 53
Unique PMIDs in link table       : 3,667
Unique PMIDs in paper terms      : 34
Cleaned patent–PMID links        : 3,773
Focal term rows                  : 0
Unique patents with focal terms  : 0
Unique focal terms               : 0


## Task 2: Measure Overlap Intensity

Count unique focal terms per patent and compute the distribution.

In [10]:
# Step 2: Count unique focal terms per patent
counts = (
    focal_terms.group_by("patent_id")
    .agg(pl.col("focal_term").n_unique().alias("num_focal_terms"))
    .sort("patent_id")
)
total = len(counts)

if total == 0:
    print("No patents with focal terms — skipping Task 2 statistics.")
    mean_val = median_val = std_val = min_val = max_val = n_one = 0
else:
    mean_val   = counts["num_focal_terms"].mean()
    median_val = counts["num_focal_terms"].median()
    std_val    = counts["num_focal_terms"].std()
    min_val    = counts["num_focal_terms"].min()
    max_val    = counts["num_focal_terms"].max()
    n_one      = int((counts["num_focal_terms"] == 1).sum())

    # Step 3: Summary statistics
    print(f"Unique patents with focal terms     : {total:,}")
    print(f"Mean focal terms per patent         : {mean_val:.2f}")
    print(f"Median                              : {median_val:.1f}")
    print(f"Std Dev                             : {std_val:.2f}")
    print(f"Min                                 : {min_val}")
    print(f"Max                                 : {max_val}")
    print(f"Patents with exactly 1 focal term   : {n_one} ({n_one/total*100:.1f}%)")
    print()
    print(counts.head(10))

No patents with focal terms — skipping Task 2 statistics.


In [11]:
# Steps 5-6: Visualizations — histogram and density/KDE plot
if total > 0:
    values = counts["num_focal_terms"].to_numpy()

    # Histogram
    fig1, ax1 = plt.subplots(figsize=(8, 5))
    ax1.hist(values, bins=range(1, int(max_val) + 2), color="steelblue",
             edgecolor="white", align="left")
    ax1.axvline(mean_val,   color="red",    linestyle="--",
                label=f"Mean ({mean_val:.2f})")
    ax1.axvline(median_val, color="orange", linestyle="--",
                label=f"Median ({median_val:.0f})")
    ax1.set_title("Histogram of Focal Terms per Patent (FullSample 5000)")
    ax1.set_xlabel("Number of Focal Terms")
    ax1.set_ylabel("Number of Patents")
    ax1.legend()
    plt.tight_layout()
    plt.savefig(VIZ_DIR / "histogram_focal_terms_fullsample_5000.png", dpi=150)
    plt.show()

    # Density / KDE plot
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    if len(values) > 1:
        kde = gaussian_kde(values)
        x = np.linspace(values.min() - 0.5, values.max() + 0.5, 300)
        ax2.plot(x, kde(x), color="steelblue")
    ax2.axvline(mean_val,   color="red",    linestyle="--",
                label=f"Mean ({mean_val:.2f})")
    ax2.axvline(median_val, color="orange", linestyle="--",
                label=f"Median ({median_val:.0f})")
    ax2.set_title("Density Plot of Focal Terms per Patent (FullSample 5000)")
    ax2.set_xlabel("Number of Focal Terms")
    ax2.set_ylabel("Density")
    ax2.legend()
    plt.tight_layout()
    plt.savefig(VIZ_DIR / "density_focal_terms_fullsample_5000.png", dpi=150)
    plt.show()

In [12]:
# Steps 4 + 7: Save per-patent counts and summary statistics
if total > 0:
    counts.write_csv(OUT_DIR / "focal_term_counts_per_patent_fullsample_5000.csv")
    print("Saved: focal_term_counts_per_patent_fullsample_5000.csv")

    summary_t2 = pl.DataFrame({
        "statistic": ["mean", "median", "std", "min", "max",
                      "n_patents", "pct_exactly_1"],
        "value": [mean_val, median_val, std_val, float(min_val), float(max_val),
                  float(total), float(n_one / total * 100)],
    })
    summary_t2.write_csv(OUT_DIR / "task2_summary_stats_fullsample_5000.csv")
    print("Saved: task2_summary_stats_fullsample_5000.csv")
    print()
    print(summary_t2)

## Task 3: Semantic Context Comparison

Assess whether focal terms are used in similar or different semantic contexts in patents vs. cited scientific papers.

**Approach:** embed bag-of-words context sentences with `sentence-transformers/all-MiniLM-L6-v2`, then compute row-wise cosine similarity.

In [13]:
# Step 3 (patent side): build patent context per (patent_id, focal_term)
# Patent context = all other terms in the same patent (excluding the focal term itself)
patent_contexts = (
    pat_terms.select(["patent_id", "term"])
    .join(focal_terms.select(["patent_id", "focal_term"]), on="patent_id", how="inner")
    .filter(pl.col("term") != pl.col("focal_term"))
    .group_by(["patent_id", "focal_term"])
    .agg(pl.col("term").alias("patent_context"))
)
print(f"Patent context rows: {len(patent_contexts)}")
patent_contexts.head(3)

Patent context rows: 0


patent_id,focal_term,patent_context
str,str,list[str]


In [14]:
# Step 2: Map (patent_id, focal_term) to cited PMIDs that contain the focal term
focal_with_pmid = (
    focal_terms.select(["patent_id", "focal_term"])
    .join(link_clean, on="patent_id", how="inner")
    .join(
        pmed_terms.rename({"term": "focal_term"}),
        on=["pmid", "focal_term"],
        how="inner",
    )
    .select(["patent_id", "focal_term", "pmid"])
    .unique()
)
print(f"(patent_id, focal_term, pmid) triples: {len(focal_with_pmid)}")

# Step 3 (paper side): all other terms from cited PMIDs that contain the focal term
paper_contexts = (
    focal_with_pmid
    .join(pmed_terms, on="pmid", how="inner")
    .filter(pl.col("term") != pl.col("focal_term"))
    .group_by(["patent_id", "focal_term"])
    .agg(pl.col("term").unique().alias("paper_context"))
)
print(f"Paper context rows: {len(paper_contexts)}")
paper_contexts.head(3)

(patent_id, focal_term, pmid) triples: 0
Paper context rows: 0


patent_id,focal_term,paper_context
str,str,list[str]


In [15]:
# Step 4: Assemble context table
context_df = (
    focal_terms.select(["patent_id", "focal_term"])
    .join(patent_contexts, on=["patent_id", "focal_term"], how="left")
    .join(paper_contexts,  on=["patent_id", "focal_term"], how="left")
)
print(context_df.shape)
context_df.head(5)

(0, 4)


patent_id,focal_term,patent_context,paper_context
str,str,list[str],list[str]


In [16]:
# Step 4 (cont.): Serialize contexts to sentences
# Format: "<focal_term> <context_term_1> <context_term_2> ..."
patent_sentences = [
    f"{row['focal_term']} {' '.join(row['patent_context'] or [])}"
    for row in context_df.iter_rows(named=True)
]
paper_sentences = [
    f"{row['focal_term']} {' '.join(row['paper_context'] or [])}"
    for row in context_df.iter_rows(named=True)
]
if patent_sentences:
    print(f"Sample patent sentence : {patent_sentences[0][:120]}")
    print(f"Sample paper  sentence : {paper_sentences[0][:120]}")

In [17]:
# Step 7: Save context table (parquet with list cols; CSV with serialized strings)
context_df.write_parquet(OUT_DIR / "focal_term_context_fullsample_5000.parquet")

context_csv = context_df.with_columns([
    pl.col("patent_context").map_elements(
        lambda x: " ".join(x) if x is not None else "", return_dtype=pl.String
    ).alias("patent_context"),
    pl.col("paper_context").map_elements(
        lambda x: " ".join(x) if x is not None else "", return_dtype=pl.String
    ).alias("paper_context"),
])
context_csv.write_csv(OUT_DIR / "focal_term_context_fullsample_5000.csv")
print("Saved: focal_term_context_fullsample_5000.parquet and .csv")

Saved: focal_term_context_fullsample_5000.parquet and .csv


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_sim

if not patent_sentences:
    print("No focal terms — skipping embedding and cosine similarity.")
    cosine_df = context_df.select(["patent_id", "focal_term"]).with_columns(
        pl.lit(None).cast(pl.Float64).alias("cosine_similarity")
    )
    print(cosine_df)
else:
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    patent_emb = model.encode(patent_sentences, show_progress_bar=True, batch_size=256)
    paper_emb  = model.encode(paper_sentences,  show_progress_bar=True, batch_size=256)

    sims = sk_cosine_sim(patent_emb, paper_emb).diagonal()

    cosine_df = context_df.select(["patent_id", "focal_term"]).with_columns(
        pl.Series("cosine_similarity", sims.astype(float))
    )
    print(cosine_df.head(10))

In [ ]:
# Step 8: Save cosine similarity results
cosine_df.write_parquet(OUT_DIR / "cosine_similarity_results_fullsample_5000.parquet")
cosine_df.write_csv(OUT_DIR / "cosine_similarity_results_fullsample_5000.csv")
print("Saved: cosine_similarity_results_fullsample_5000.parquet and .csv")

In [ ]:
sim = cosine_df["cosine_similarity"].drop_nulls().to_numpy()

if len(sim) == 0:
    print("No cosine similarity results — skipping summary statistics and visualization.")
    pl.DataFrame({
        "statistic": ["mean", "median", "std", "min", "max"],
        "value": [None, None, None, None, None],
    }).write_csv(OUT_DIR / "task3_cosine_summary_stats_fullsample_5000.csv")
    print("Saved: task3_cosine_summary_stats_fullsample_5000.csv (empty)")
else:
    mean_sim   = float(sim.mean())
    median_sim = float(np.median(sim))
    std_sim    = float(sim.std())
    min_sim    = float(sim.min())
    max_sim    = float(sim.max())

    print("=== Cosine Similarity Summary Statistics ===")
    print(f"  Mean   : {mean_sim:.3f}")
    print(f"  Median : {median_sim:.3f}")
    print(f"  Std    : {std_sim:.3f}")
    print(f"  Min    : {min_sim:.3f}")
    print(f"  Max    : {max_sim:.3f}")

    pl.DataFrame({
        "statistic": ["mean", "median", "std", "min", "max"],
        "value": [mean_sim, median_sim, std_sim, min_sim, max_sim],
    }).write_csv(OUT_DIR / "task3_cosine_summary_stats_fullsample_5000.csv")
    print("Saved: task3_cosine_summary_stats_fullsample_5000.csv")

    plt.figure(figsize=(8, 4))
    plt.hist(sim, bins=20, color="steelblue", edgecolor="black")
    plt.axvline(mean_sim,   color="red",    linestyle="--",
                label=f"Mean ({mean_sim:.3f})")
    plt.axvline(median_sim, color="orange", linestyle="--",
                label=f"Median ({median_sim:.3f})")
    plt.xlabel("Cosine Similarity")
    plt.ylabel("Count")
    plt.title(
        "Distribution of Semantic Similarity\n"
        "(FullSample 5000 — patent vs. paper context per focal term)"
    )
    plt.legend()
    plt.tight_layout()
    plt.savefig(VIZ_DIR / "cosine_similarity_distribution_fullsample_5000.png", dpi=150)
    plt.show()

## Final Summary

This notebook tested the Tasks 1–3 pipeline on the first **5,000 rows** of each FullSampleGloria dataset.

**Outputs produced** (`output/fullsample_outputs/`):
| File | Task |
|------|------|
| `focal_terms_fullsample_5000.parquet/.csv` | Task 1 |
| `focal_term_counts_per_patent_fullsample_5000.csv` | Task 2 |
| `task2_summary_stats_fullsample_5000.csv` | Task 2 |
| `focal_term_context_fullsample_5000.parquet/.csv` | Task 3 |
| `cosine_similarity_results_fullsample_5000.parquet/.csv` | Task 3 |
| `task3_cosine_summary_stats_fullsample_5000.csv` | Task 3 |

**Visualizations** (`visualizations/fullsample_visualizations/`):
- `histogram_focal_terms_fullsample_5000.png`
- `density_focal_terms_fullsample_5000.png`
- `cosine_similarity_distribution_fullsample_5000.png`

> **Note:** If zero focal terms are found, this is expected due to row slicing across three large files — it does not indicate a pipeline failure.